# Check · SMILES chunk pooling

Verifies the change in `ddi.data.chunk_and_embed`: long SMILES (> 256 characters) are now embedded as the mean of the `[CLS]` vectors of all chunks; SMILES of ≤ 256 characters are unchanged. The thesis graph used only the first chunk.

Needs the META-DDIE databases (downloaded once, about 21 MB) and the ChemBERTa-77M-MLM weights from Hugging Face (downloaded once).

In [ ]:
import os
REPO_DIR = os.path.abspath(os.path.join(os.path.dirname(__file__), "..", ".."))
DATA_ROOT = "<DATA_ROOT: e.g. /home/you/ddi_data>"
WORK_ROOT = "<WORK_ROOT: e.g. /home/you/ddi_runs>"

In [ ]:
import os, sys
if os.path.join(REPO_DIR, "src") not in sys.path:
    sys.path.insert(0, os.path.join(REPO_DIR, "src"))

import torch
from transformers import AutoTokenizer, AutoModel
from ddi.paths import get_paths, RAW_DATA_GDRIVE_ID
from ddi.data import download_raw_data, load_data_into_dataframe, get_all_events, chunk_and_embed

paths = get_paths(DATA_ROOT, WORK_ROOT)
drug_db, event_db = download_raw_data(paths.raw_data_dir, RAW_DATA_GDRIVE_ID)
tok = AutoTokenizer.from_pretrained("DeepChem/ChemBERTa-77M-MLM")
mdl = AutoModel.from_pretrained("DeepChem/ChemBERTa-77M-MLM").eval()

## How many drugs and interactions are affected

In [ ]:
df = load_data_into_dataframe(drug_db, "drug")
lengths = df["smile"].str.len()
long_mask = lengths > 256
print(f"{int(long_mask.sum())} of {len(df)} drugs have SMILES > 256 characters "
      f"(longest {int(lengths.max())}, median {int(lengths.median())})")

ev = get_all_events(event_db)
ev = ev[ev["label"].astype(int) <= 227]
long_ids = set(df.loc[long_mask, "id"])
hit = ev["id1"].isin(long_ids) | ev["id2"].isin(long_ids)
print(f"{int(hit.sum()):,} of {len(ev):,} interactions ({100*hit.mean():.2f}%) involve a long-SMILES drug")

## New embedding vs thesis behaviour (first chunk only)

In [ ]:
def first_chunk_only(s):
    """The thesis behaviour, for comparison."""
    with torch.no_grad():
        return mdl(**tok(s[:256], return_tensors="pt")).last_hidden_state[:, 0, :]

short_smiles = df.loc[~long_mask, "smile"].tolist()[:50]
long_smiles = df.loc[long_mask, "smile"].tolist()

same = all(torch.equal(chunk_and_embed(s, tok, mdl), first_chunk_only(s)) for s in short_smiles)
print(f"short SMILES (first 50) bit-identical to the thesis embedding: {same}")

cos = torch.nn.functional.cosine_similarity
sims = [float(cos(chunk_and_embed(s, tok, mdl), first_chunk_only(s))) for s in long_smiles]
print(f"long SMILES ({len(sims)}): cosine(new, thesis) min {min(sims):.3f} | "
      f"mean {sum(sims)/len(sims):.3f} | max {max(sims):.3f}")
print("shape:", tuple(chunk_and_embed(long_smiles[0], tok, mdl).shape))